# Exercice 1 — Provision, portefeuille et distances : l'algèbre linéaire de l'actuaire vie

## Contexte métier
Vous appartenez à l'équipe Provisionnement Épargne/Retraite. Un portefeuille de **rentes temporaires de 5 ans** (assurés de 60 ans) doit être valorisé, choqué en taux, puis les contrats doivent être comparés pour une future segmentation. Vous allez constater qu'une **provision est un produit scalaire**, qu'un **portefeuille est une matrice**, et que **l'échelle des variables change les distances**.

## Objectifs
- Écrire une provision comme produit scalaire `Σ kpx · F · v^k` (vecteurs NumPy)
- Valoriser tout un portefeuille et plusieurs scénarios de taux avec un produit matriciel
- Mesurer la distance entre contrats avant / après standardisation (μ, σ du **train**)
- Lire une matrice de covariance / corrélation et ses valeurs propres (intuition PCA)

## Rappels théoriques mobilisés (slides de la session)
- *Vecteurs et matrices : le langage des données* — produit scalaire, produit matriciel, norme et distance
- *Covariance et corrélation* — signe, force, piège « corrélation nulle ≠ indépendance »
- *Valeurs et vecteurs propres* — variance portée par chaque direction
- *Normalisation : pourquoi et comment* — μ, σ estimés sur le train uniquement

## Mini-dictionnaire & prérequis
| Élément | Signification / usage |
|---|---|
| `q` | taux de mortalité annuels q_x, q_{x+1}… (table **fictive**) |
| `survie` (kpx) | probabilité d'être vivant à la fin de l'année k : produit cumulé des (1 − q) |
| `v` / facteur d'actualisation | v = 1/(1+i) ; flux de l'année k actualisé : v^k |
| `np.cumprod`, `np.dot`, `@` | produit cumulé ; produit scalaire ; produit matriciel |
| `np.linalg.norm(x-y)` | norme euclidienne ‖x − y‖₂ = distance entre deux contrats |
| `df.mean()`, `df.std()`, `df.cov()`, `df.corr()` | μ, σ, covariance, corrélation d'un DataFrame |
| `np.linalg.eigh(M)` | valeurs et vecteurs propres d'une matrice symétrique (ordre croissant !) |

## Mode d'emploi
- Complétez les cellules marquées `...  # À COMPLÉTER` puis exécutez-les **dans l'ordre**.
- Les cellules **Vérification** vous disent immédiatement si vous êtes sur la bonne piste (✅ / ❌ + indice).
- Bloqué ? Ouvrez les **💡 indices** progressifs avant de demander la correction.
- À la fin, la cellule `bilan()` donne votre score.

In [ ]:
# Outil d'auto-vérification — NE PAS MODIFIER
import numpy as np, pandas as pd, matplotlib.pyplot as plt, hashlib
_resultats = {}
def verifier(nom, condition, indice=""):
    """Évalue `condition` (une fonction sans argument) ; affiche ✅ ou ❌ + indice."""
    try: ok = bool(condition())
    except Exception: ok = False
    _resultats[nom] = ok
    print(("✅ " if ok else "❌ ") + nom + ("" if ok or not indice else "\n    💡 " + indice))
def bilan():
    n, k = len(_resultats), sum(_resultats.values())
    print(f"\n🏁 Score : {k}/{n} vérifications réussies")
    print("   Bravo, exercice terminé !" if k == n and n > 0 else "   Reprenez les ❌ ci-dessus (relancez la cellule concernée puis `bilan()`).")
def _h(s): return hashlib.sha1(s.encode()).hexdigest()[:8]
%matplotlib inline

---
## Partie A — Une provision est un produit scalaire

Une rente annuelle de **12 000 €** est versée en fin d'année tant que l'assuré est vivant, pendant 5 ans. Table de mortalité **fictive** (ages 60 à 64) et taux technique 2 %.

In [ ]:
q = np.array([0.0085, 0.0093, 0.0102, 0.0112, 0.0123])   # q_60 ... q_64 (table fictive)
F = np.array([12000., 12000., 12000., 12000., 12000.])   # flux versés en fin d'année k = 1..5
taux = 0.02
k = np.arange(1, 6)                                      # [1, 2, 3, 4, 5]

**Q1.** Construisez (a) le vecteur d'actualisation `actualisation` = v^k, (b) le vecteur de survie `survie` (kpx = ∏(1 − q_j)), puis (c) la provision `PM = Σ kpx · F_k · v^k` **sous forme de produit scalaire**.

<details><summary>💡 Indice 1</summary>

`v = 1/(1+taux)` puis `v**k` fonctionne directement sur le vecteur `k`.

</details>

<details><summary>💡 Indice 2</summary>

kpx = (1−q₀)(1−q₁)… : c'est un **produit cumulé** → `np.cumprod(1 - q)`.

</details>

<details><summary>💡 Indice 3</summary>

Un produit scalaire de deux vecteurs : `np.dot(a, b)` ; ici `a = F * survie` (flux espérés) et `b = actualisation`.

</details>

In [ ]:
v = ...                    # À COMPLÉTER : facteur d'actualisation annuel
actualisation = ...        # À COMPLÉTER : vecteur v**k
survie = ...               # À COMPLÉTER : vecteur kpx (k=1..5)
PM = ...                   # À COMPLÉTER : provision (produit scalaire)
print(f"Provision = {PM:,.2f} €")

In [ ]:
verifier("Q1a · actualisation[0] = v", lambda: np.isclose(actualisation[0], 1/1.02), "v^1 = 1/(1+taux)")
verifier("Q1a · taille du vecteur d'actualisation", lambda: len(actualisation) == 5)
verifier("Q1b · survie décroissante, survie[0] = 1-q[0]", lambda: np.isclose(survie[0], 1-q[0]) and np.all(np.diff(survie) < 0), "utilisez np.cumprod(1 - q)")
verifier("Q1c · provision correcte", lambda: np.isclose(PM, 54966.808571, atol=0.5), "PM = somme de F_k · kpx · v^k")


## Partie B — Un portefeuille est une matrice

Le portefeuille contient 6 contrats (de rentes différentes, certains plus courts). Chaque ligne de `F_ptf` est un contrat, chaque colonne une année.

In [ ]:
F_ptf = np.array([
    [12000, 12000, 12000, 12000, 12000],
    [ 8000,  8000,  8000,  8000,  8000],
    [20000, 20000, 20000,     0,     0],
    [ 5000,  5000,  5000,  5000,  5000],
    [15000, 15000, 15000, 15000,     0],
    [ 9000,  9000,  9000,  9000,  9000]], dtype=float)
print("Forme de F_ptf :", F_ptf.shape)

**Q2.** (a) Calculez en **une seule opération** le vecteur `PM_ptf` des 6 provisions (indice : un vecteur de poids `w = kpx · v^k`, puis `F_ptf @ w`). (b) Calculez `PM_total`.

<details><summary>💡 Indice 1</summary>

Chaque contrat a la même table de mortalité et le même taux : le même vecteur de poids `w` s'applique à toutes les lignes.

</details>

<details><summary>💡 Indice 2</summary>

`w = survie * actualisation` ; `PM_ptf = F_ptf @ w` ; `PM_total = PM_ptf.sum()`.

</details>

In [ ]:
w = ...          # À COMPLÉTER : vecteur de poids (taille 5)
PM_ptf = ...     # À COMPLÉTER : vecteur des 6 provisions
PM_total = ...   # À COMPLÉTER
print(np.round(PM_ptf, 0), f"\nTotal = {PM_total:,.0f} €")

In [ ]:
verifier("Q2 · PM_ptf a 6 composantes", lambda: np.shape(PM_ptf) == (6,))
verifier("Q2 · cohérence avec Q1 (contrat n°1)", lambda: np.isclose(PM_ptf[0], PM), "la ligne 0 de F_ptf est le contrat de la partie A")
verifier("Q2 · PM_total correct", lambda: np.isclose(PM_total, 268194.848765, atol=1))

**Q3 · Scénarios de taux.** Le contrôle des risques demande la PM sous 4 taux techniques : 1 %, 2 %, 3 %, 4 %. Construisez la matrice `W` (5 années × 4 scénarios), puis `PM_scen = F_ptf @ W` (6 contrats × 4 scénarios), le total par scénario `PM_tot_scen`, et la sensibilité `sensibilite` = variation relative de PM_total vs scénario central (2 %).

<details><summary>💡 Indice 1</summary>

Utilisez le *broadcasting* : `survie[:, None]` est une colonne (5×1) et `(1 + taux_scen[None, :])` une ligne (1×4).

</details>

<details><summary>💡 Indice 2</summary>

`W = survie[:, None] * (1 + taux_scen[None, :]) ** (-k[:, None])`

</details>

<details><summary>💡 Indice 3</summary>

`PM_tot_scen = PM_scen.sum(axis=0)` ; `sensibilite = PM_tot_scen / PM_tot_scen[1] - 1`

</details>

In [ ]:
taux_scen = np.array([0.01, 0.02, 0.03, 0.04])
W = ...             # À COMPLÉTER : matrice 5 x 4
PM_scen = ...       # À COMPLÉTER : matrice 6 x 4
PM_tot_scen = ...   # À COMPLÉTER : vecteur de 4 totaux
sensibilite = ...   # À COMPLÉTER : variation relative vs scénario central
print(pd.Series(np.round(sensibilite*100, 2), index=[f"{t:.0%}" for t in taux_scen]).rename("Δ PM (%)"))

In [ ]:
verifier("Q3 · W de forme (5, 4)", lambda: W.shape == (5, 4))
verifier("Q3 · PM_scen de forme (6, 4)", lambda: PM_scen.shape == (6, 4))
verifier("Q3 · colonne 2 % = PM_ptf", lambda: np.allclose(PM_scen[:, 1], PM_ptf), "le scénario 2 % doit redonner la partie B")
verifier("Q3 · une hausse des taux baisse la PM", lambda: np.all(np.diff(PM_tot_scen) < 0))
verifier("Q3 · sensibilités correctes", lambda: np.allclose(sensibilite, np.array([0.026409, 0.0, -0.025322, -0.049616]), atol=1e-4))

---
## Partie C — Distance entre contrats et normalisation

Pour une future segmentation, on mesure la ressemblance entre contrats avec la distance euclidienne. Voici un portefeuille d'entraînement simulé (`train`, 400 contrats) et trois contrats à comparer.

In [ ]:
rng = np.random.default_rng(42)
n = 400
age = np.clip(rng.normal(52, 12, n), 25, 85).round()
anciennete = np.clip(0.25 * (age - 25) + rng.normal(0, 4, n), 0.5, 35).round(1)
encours = np.exp(9.9 + 0.012 * (age - 52) + 0.4 * rng.normal(size=n)).round(-2)
versements = (0.03 * encours * np.exp(rng.normal(0, 0.3, n))).round(-1)
train = pd.DataFrame({"age": age, "encours": encours, "anciennete": anciennete, "versements": versements})
cols = ["age", "encours", "anciennete"]
a = np.array([45, 50000, 10.])   # contrat A
b = np.array([46, 52000,  2.])   # contrat B : même âge, +2 000 € d'encours, mais 8 ans d'ancienneté de moins
c = np.array([70, 50500, 10.])   # contrat C : 25 ans de plus, mais encours et ancienneté quasi identiques à A
train.describe().round(0).loc[["mean", "std"]]

**Q4.** (a) Calculez la distance brute `d_ab`, `d_ac`. (b) Estimez `mu` et `sigma` **sur `train[cols]` uniquement**, standardisez a, b, c et recalculez `d_ab_z`, `d_ac_z`. (c) Indiquez dans `plus_proche_brut` et `plus_proche_std` (`"B"` ou `"C"`) le contrat le plus proche de A.

<details><summary>💡 Indice 1</summary>

`np.linalg.norm(a - b)` donne la distance euclidienne.

</details>

<details><summary>💡 Indice 2</summary>

`mu = train[cols].mean().values` et `sigma = train[cols].std().values` ; `a_z = (a - mu) / sigma`.

</details>

In [ ]:
d_ab = ...   # À COMPLÉTER
d_ac = ...   # À COMPLÉTER
mu = ...     # À COMPLÉTER : moyennes du TRAIN (vecteur de 3)
sigma = ...  # À COMPLÉTER : écarts-types du TRAIN
a_z, b_z, c_z = ..., ..., ...    # À COMPLÉTER : contrats standardisés
d_ab_z = ...
d_ac_z = ...
plus_proche_brut = ...   # "B" ou "C"
plus_proche_std = ...    # "B" ou "C"
print(f"Brut : d(A,B)={d_ab:,.1f}  d(A,C)={d_ac:,.1f}\nStd  : d(A,B)={d_ab_z:.2f}  d(A,C)={d_ac_z:.2f}")

In [ ]:
verifier("Q4a · distances brutes", lambda: np.isclose(d_ab, 2000.01625, atol=0.1) and np.isclose(d_ac, 500.62461, atol=0.1))
verifier("Q4b · μ et σ estimés sur le train", lambda: np.allclose(mu, train[cols].mean().values) and np.allclose(sigma, train[cols].std().values, atol=1e-6))
verifier("Q4b · distances standardisées", lambda: np.isclose(d_ab_z, 1.795408, atol=0.01) and np.isclose(d_ac_z, 2.195417, atol=0.01))
verifier("Q4c · le plus proche change après standardisation", lambda: (plus_proche_brut, plus_proche_std) == ('C', 'B'), "comparez d_ab et d_ac dans les deux cas")

**Q4d · QCM.** Pourquoi estime-t-on μ et σ sur le train uniquement ?

- **A.** Pour que le calcul soit plus rapide
- **B.** Pour éviter qu'une information du test (futur) ne fuite dans la préparation des données
- **C.** Parce que le test n'a pas de variance

Renseignez `reponse_q4d` (`"A"`, `"B"` ou `"C"`).

In [ ]:
reponse_q4d = ...   # À COMPLÉTER

In [ ]:
verifier("Q4d · fuite de données", lambda: reponse_q4d == "B", "relisez l'encadré « Attention » de la slide Normalisation")

---
## Partie D — Covariance, corrélation et valeurs propres

**Q5.** (a) Calculez `cov = train.cov()` et `corr = train.corr()` (4 variables). (b) Calculez les valeurs propres de `corr` avec `np.linalg.eigh`, triées par ordre **décroissant** → `vp`. (c) `part_pc1` = part de variance expliquée par la 1ʳᵉ composante ; `n_pc_85` = nombre minimal de composantes pour expliquer ≥ 85 % de la variance.

<details><summary>💡 Indice 1</summary>

`eigh` renvoie les valeurs propres en ordre **croissant** : retournez-les avec `[::-1]`.

</details>

<details><summary>💡 Indice 2</summary>

Part de variance : `vp / vp.sum()` ; cumul : `np.cumsum(...)` ; `n_pc_85 = int(np.argmax(cumul >= 0.85)) + 1`.

</details>

In [ ]:
cov = ...    # À COMPLÉTER
corr = ...   # À COMPLÉTER
vp = ...     # À COMPLÉTER : valeurs propres décroissantes de corr
part_pc1 = ...   # À COMPLÉTER
n_pc_85 = ...    # À COMPLÉTER
print(corr.round(2)); print("Valeurs propres :", np.round(vp, 2), f"| PC1 = {part_pc1:.0%} | composantes pour 85 % : {n_pc_85}")

In [ ]:
verifier("Q5a · corr symétrique, diagonale = 1", lambda: np.allclose(corr.values, corr.values.T) and np.allclose(np.diag(corr.values), 1))
verifier("Q5b · valeurs propres décroissantes", lambda: np.all(np.diff(vp) <= 0))
verifier("Q5b · somme des valeurs propres = nb de variables (trace)", lambda: np.isclose(vp.sum(), 4), "la trace d'une matrice de corrélation 4×4 vaut 4")
verifier("Q5c · part de variance de PC1", lambda: np.isclose(part_pc1, 0.499123, atol=1e-3))
verifier("Q5c · nombre de composantes pour 85 %", lambda: n_pc_85 == 3)

**Q6 · Piège classique (bonus).** Soit `x` entre −3 et 3 et `y = x²` (dépendance parfaite). Calculez `corr_xy = np.corrcoef(x, y)[0, 1]`, puis répondez : `reponse_q6` = `"A"` (« corrélation ≈ 0 donc x et y sont indépendantes ») ou `"B"` (« corrélation ≈ 0 mais y dépend entièrement de x »).

In [ ]:
x = np.linspace(-3, 3, 201)
y = x ** 2
corr_xy = ...        # À COMPLÉTER
reponse_q6 = ...     # À COMPLÉTER
print(corr_xy)

In [ ]:
verifier("Q6 · corrélation ≈ 0", lambda: abs(corr_xy) < 1e-10)
verifier("Q6 · corrélation nulle ≠ indépendance", lambda: reponse_q6 == "B")

In [ ]:
bilan()